<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Constant006.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🌀 Constant006 — The Bifurcation Map
# Objective: Map Λ emergence across ξ imbalance and noise amplitude

import numpy as np
import matplotlib.pyplot as plt

# --- Parameter Grid ---
ξ_values = np.arange(0.01, 0.10, 0.02)
noise_values = np.arange(1e-3, 1.05e-2, 5e-3)
size, steps = 100, 300
damping = 0.01
sequestration_enabled = True

# --- Result Grid ---
result_grid = np.zeros((len(ξ_values), len(noise_values)))

# --- Simulation Loop ---
for i, ξ in enumerate(ξ_values):
    for j, noise_amp in enumerate(noise_values):
        entropy = np.random.rand(size, size)
        curvature = np.zeros((size, size))

        for t in range(steps):
            noise = noise_amp * np.random.randn(size, size)
            grad_x, grad_y = np.gradient(entropy)
            grad_x = np.clip(grad_x, -1e3, 1e3)
            grad_y = np.clip(grad_y, -1e3, 1e3)
            flow = grad_x**2 + grad_y**2
            curvature += ξ * flow - damping * curvature + noise
            curvature = np.nan_to_num(curvature, nan=0.0, posinf=1e6, neginf=0.0)
            if sequestration_enabled:
                threshold = np.nanpercentile(curvature, 95)
                curvature = np.clip(curvature, 0, threshold)
            entropy += curvature * damping

        # Record final residual energy
        result_grid[i, j] = np.nanmean(curvature)

# --- Plot Heatmap ---
plt.figure(figsize=(8, 6))
plt.imshow(result_grid, origin='lower', cmap='viridis', aspect='auto',
           extent=[noise_values[0], noise_values[-1], ξ_values[0], ξ_values[-1]])
plt.colorbar(label='Residual Energy (Λ proxy)')
plt.xlabel('Noise Amplitude')
plt.ylabel('ξ Imbalance')
plt.title('Λ Emergence Bifurcation Map — Constant006')
plt.grid(False)
plt.show()
